In [1]:
import pandas as pd
import sqlite3

In [12]:
data = {
    "order_id":[1001,1002,1003,1004,1005],
    "order_date":["2024-01-01","2024-01-02","2024-01-02","2024-01-03","2024-01-03"],
    "customer_name":["Arun","Priya","Karthik","Divya","Rahul"],
    "city":["Chennai","Coimbatore","Madurai","Salem","Tirupur"],
    "product_category":["Electronics","Clothing","Grocery","Electronics","Clothing"],
    "product_name":["Headphones","Hoodie","Rice Bag","Mouse","T-Shirt"],
    "quantity":[2,1,3,2,4],
    "price":[1500,2200,900,700,600],
    "payment_method":["UPI","Card","Cash","UPI","Card"],
    "rating":[4,5,4,3,4]
}

df = pd.DataFrame(data)
df

,order_id,order_date,customer_name,city,product_category,product_name,quantity,price,payment_method,rating
0,1001,2024-01-01,Arun,Chennai,Electronics,Headphones,2,1500,UPI,4
1,1002,2024-01-02,Priya,Coimbatore,Clothing,Hoodie,1,2200,Card,5
2,1003,2024-01-02,Karthik,Madurai,Grocery,Rice Bag,3,900,Cash,4
3,1004,2024-01-03,Divya,Salem,Electronics,Mouse,2,700,UPI,3
4,1005,2024-01-03,Rahul,Tirupur,Clothing,T-Shirt,4,600,Card,4


In [13]:
conn = sqlite3.connect("online_orders.db")

df.to_sql("online_orders",
    conn,
    if_exists="replace",
    index=False
)

5

In [ ]:
def run_query(query, connection_string):
    return pd.read_sql(query, conn)

1) Find the top 3 cities generating the highest revenue.

In [15]:
query1 = """
SELECT city,
       SUM(quantity * price) AS revenue
FROM online_orders
GROUP BY city
ORDER BY revenue DESC
LIMIT 3;
"""

result1 = run_query(query1, "")
result1

,city,revenue
0,Chennai,3000
1,Madurai,2700
2,Tirupur,2400


2) Find customers whose total purchase amount is above the average customer spending.

In [16]:
query2 = """
SELECT customer_name,
       SUM(quantity * price) AS total_spent
FROM online_orders
GROUP BY customer_name
HAVING total_spent >
(
    SELECT AVG(customer_total)
    FROM
    (
        SELECT SUM(quantity * price) AS customer_total
        FROM online_orders
        GROUP BY customer_name
    )
);"""
result2 = run_query(query2, "")
result2

,customer_name,total_spent
0,Arun,3000
1,Karthik,2700
2,Rahul,2400


3) Find the most sold product category based on quantity.

In [17]:
query3 = """
SELECT product_category,
       SUM(quantity) AS total_quantity
FROM online_orders
GROUP BY product_category
ORDER BY total_quantity DESC
LIMIT 1;"""

result3 = run_query(query3, "")
result3

,product_category,total_quantity
0,Clothing,5


4) Calculate cumulative daily sales ordered by order date.

In [18]:
query4 = """
SELECT order_date,
       SUM(quantity * price) AS daily_sales,

       SUM(SUM(quantity * price))
       OVER(ORDER BY order_date) AS cumulative_sales

FROM online_orders
GROUP BY order_date;"""

result4 = run_query(query4, "")
result4

,order_date,daily_sales,cumulative_sales
0,2024-01-01,3000,3000
1,2024-01-02,4900,7900
2,2024-01-03,3800,11700


5) Rank products within each category based on price.

In [19]:
query5 = """
SELECT product_category,
       product_name,
       price,

       RANK() OVER(
           PARTITION BY product_category
           ORDER BY price DESC
       ) AS product_rank

FROM online_orders;"""
result5 = run_query(query5, "")
result5

,product_category,product_name,price,product_rank
0,Clothing,Hoodie,2200,1
1,Clothing,T-Shirt,600,2
2,Electronics,Headphones,1500,1
3,Electronics,Mouse,700,2
4,Grocery,Rice Bag,900,1


6) Find percentage contribution of each category to total revenue.

In [20]:
query6 = """
SELECT product_category,
       SUM(quantity * price) AS category_sales
FROM online_orders
GROUP BY product_category;
"""

result6 = run_query(query6, "")
result6

,product_category,category_sales
0,Clothing,4600
1,Electronics,4400
2,Grocery,2700


7) Identify customers who made purchases on consecutive days.

In [21]:
query7 = """
SELECT customer_name,
       order_date
FROM online_orders
ORDER BY customer_name, order_date;
"""
result7 = run_query(query7, "")
result7

,customer_name,order_date
0,Arun,2024-01-01
1,Divya,2024-01-03
2,Karthik,2024-01-02
3,Priya,2024-01-02
4,Rahul,2024-01-03
